In [ ]:
# Notebook working directory helper (avoids hard-coded %cd)
from __future__ import annotations

import os
from pathlib import Path

cwd = Path.cwd()
if (cwd / "fourier_cppn.py").exists():
    repo_root = cwd
elif (cwd / "latent-terrain-pytorch" / "fourier_cppn.py").exists():
    repo_root = cwd / "latent-terrain-pytorch"
    os.chdir(repo_root)

print(f"cwd = {Path.cwd()}")

# Tuning / Spatial CPPN Experiments

This notebook explores mapping **2D control-space trajectories** (packed Bézier curves) to **audio latent trajectories** using a Fourier-CPPN.

Run cells top-to-bottom. Parameters are grouped near the top.

## 1) Imports & Helpers

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset

from pipeline import audio as audio_utils
from pipeline import train as train_utils
from scripts.packed_bezier_curves import generate_packed_bezier_curves
from scripts.run_pipeline import (
    CPPNHparams,
    RunConfig,
    build_codec,
    pick_device,
    segments_to_latent_trajectories,
)

In [ ]:
class TrajectoryDataset2D(Dataset):
    """Pairs 2D coords with latent vectors.

    Args:
        coords_xy: (N, 2) or (1, 2, N)
        latent_td: (N, D) or (1, D, N)

    Internally stored as:
        self.coords: (N, 2)
        self.traj:   (N, D)
    """

    def __init__(self, coords_xy: torch.Tensor, latent_td: torch.Tensor):
        if latent_td.dim() == 3:
            traj_td = latent_td.squeeze(0).transpose(0, 1).contiguous()  # (N, D)
        elif latent_td.dim() == 2:
            traj_td = latent_td
        else:
            raise ValueError("latent_td must be (1,D,N) or (N,D)")

        if coords_xy.dim() == 3:
            coords_xy = coords_xy.squeeze(0).transpose(0, 1).contiguous()  # (N, 2)
        elif coords_xy.dim() == 2:
            coords_xy = coords_xy
        else:
            raise ValueError("coords_xy must be (1,2,N) or (N,2)")

        n = min(coords_xy.shape[0], traj_td.shape[0])
        if coords_xy.shape[0] != traj_td.shape[0]:
            print(
                f"Warning: length mismatch coords={coords_xy.shape[0]} traj={traj_td.shape[0]}; "
                f"truncating to n={n}."
            )
        self.coords = coords_xy[:n]
        self.traj = traj_td[:n]

    def __len__(self) -> int:
        return int(self.traj.shape[0])

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.coords[idx], self.traj[idx]


def plot_curves(
    curves: List[torch.Tensor],
    *,
    max_curves: int,
    spacing: float,
    clearance: float,
    tile_half: float,
    anchors: np.ndarray | None = None,
    plot_anchors: bool = True,
    plot_bboxes: bool = True
):
    fig, ax = plt.subplots(figsize=(6.2, 6.2))
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)

    # Unit square
    from matplotlib.patches import Rectangle
    ax.add_patch(Rectangle((0, 0), 1, 1, fill=False, linestyle="--", edgecolor="gray", linewidth=1.0))

    cmap = plt.get_cmap("tab20")
    colors = cmap(np.linspace(0, 1, max(1, len(curves))))
    for i, pts in enumerate(curves):
        p = pts.detach().cpu().numpy()
        ax.plot(p[:, 0], p[:, 1], "-", lw=1.2, color=colors[i % len(colors)])
        stride = max(1, pts.shape[0] // 50)
        ax.scatter(p[::stride, 0], p[::stride, 1], s=6, color=colors[i % len(colors)], alpha=0.7)

    if anchors is not None and anchors.size > 0 and plot_anchors:
        ax.scatter(anchors[:, 0], anchors[:, 1], s=10, c="black", alpha=0.3, label="anchors")
        ax.legend(loc="best")

    if plot_bboxes and anchors is not None and anchors.size > 0:
        first = True
        for (axc, ayc) in anchors:
            x0 = max(0.0, float(axc) - tile_half)
            y0 = max(0.0, float(ayc) - tile_half)
            x1 = min(1.0, float(axc) + tile_half)
            y1 = min(1.0, float(ayc) + tile_half)
            rect = Rectangle(
                (x0, y0),
                x1 - x0,
                y1 - y0,
                fill=False,
                linestyle=":",
                edgecolor="black",
                linewidth=0.8,
                alpha=0.4,
                label="bboxes" if first else None,
            )
            ax.add_patch(rect)
            first = False
        ax.legend(loc="best")

    title = f"Packed Bézier curves (placed {len(curves)}/{max_curves}), spacing={spacing}, clearance={clearance}"
    ax.set_title(title)
    plt.show()

## 2) Generate 2D Control Trajectories (Packed Bézier Curves)

In [ ]:
# Curve packing params
max_curves = 36
n_points_per_curve = 22
spacing = 0.03
anchor_radius = 0.14
tile_half = 0.7
clearance = 0.06
seed = 549824
max_attempts = 1000

curves, metas, anchors = generate_packed_bezier_curves(
    max_curves=max_curves,
    n_points_per_curve=n_points_per_curve,
    spacing=spacing,
    anchor_radius=anchor_radius,
    tile_half=tile_half,
    clearance=clearance,
    oversample=512,
    seed=seed,
    device="cpu",
    check_stride=3,
    max_attempts=max_attempts,
    reject_short=True,
 )

plot_curves(
    curves,
    anchors=anchors,
    max_curves=max_curves,
    spacing=spacing,
    clearance=clearance,
    tile_half=tile_half,
    plot_anchors=False,
    plot_bboxes=False,
 )

## 3) Load Audio Segments + Encode to Latent Trajectories

In [ ]:
# Dataset/codec config
config = RunConfig(
    dataset_dir="data/aam_test",
    dataset_name="drums",
    codec_name="sao",
    out_dir="outputs/drums",
    segment_seconds=1,
    target_sr=44100,
    stereo=False,
    save_models=False,
    save_audio=True,
    # Use "cuda" if available; otherwise try "mps" (macOS) or "cpu".
    device="mps",
    hpo_trials=30,
    hpo_epochs=400,
 )

# CPPN training defaults (used below)
hparams = CPPNHparams(
    gauss_scale=float(1.0),
    epochs=int(5000),
    batch_size=int(64),
    c_max=int(512),
    mapping_size=int(256),
    lr=float(1e-3),
 )

max_segments = 8  # number of audio segments to load/encode

In [ ]:
device = pick_device(config.device)
print(f"device = {device}")

audio_dir = Path(config.dataset_dir)
if not audio_dir.exists():
    raise FileNotFoundError(f"Audio directory not found: {audio_dir}")

segment_samples = int(round(config.segment_seconds * config.target_sr))
segments: List[torch.Tensor] = audio_utils.build_segment_dataset(
    audio_dir,
    target_sr=config.target_sr,
    segment_samples=segment_samples,
    use_stereo=config.stereo,
    max_segments=max_segments,
 )
if len(segments) == 0:
    raise RuntimeError(f"No valid segments found in {audio_dir}")
print(f"segments: {len(segments)}")

codec = build_codec(config.codec_name, device=device)

# Shuffle segments deterministically (optional)
gen = torch.Generator().manual_seed(4)
idx = torch.randperm(len(segments), generator=gen).tolist()
segments = [segments[i] for i in idx]

In [ ]:
print(f"{len(curves)} curves generated; each has shape {tuple(curves[0].shape)}")

In [ ]:
# Encode segments to latent trajectories: list of (1, D, T_lat) tensors
trajectories = segments_to_latent_trajectories(segments[: len(curves)], codec, device=device)
print(f"trajectories: {len(trajectories)}")

In [ ]:
trajectories[0].shape  # (1, D, T_lat)

In [ ]:
# Concatenate along time to make one long latent stream (T_total, D)
print(f"loaded {len(trajectories)} trajectories; first shape={tuple(trajectories[0].shape)}")
latent_stream_td = torch.cat(trajectories, dim=2).squeeze(0).transpose(0, 1).contiguous()
latent_stream_td.shape  # (T_total, D)

In [ ]:
# Listen to a segment (optional)
from IPython.display import Audio, display

display(Audio(segments[2].cpu().numpy()[0], rate=config.target_sr))

## 4) Build Training Dataset (2D coords → latent vector)

In [ ]:
# Flatten curves into a single (N_coords, 2) coordinate stream
coords_stream_xy = torch.cat(curves, dim=0).contiguous()
coords_stream_xy.shape

In [ ]:
# Build dataset and dataloader
ds = TrajectoryDataset2D(coords_stream_xy, latent_stream_td)
dl = DataLoader(
    ds,
    batch_size=min(int(hparams.batch_size), len(ds)),
    shuffle=True,
    drop_last=False,
 )

print(f"dataset length={len(ds)}; batch={dl.batch_size}")

In [ ]:
xb, yb = next(iter(dl))
print(f"coords batch: {tuple(xb.shape)}  |  latent batch: {tuple(yb.shape)}")

## 5) Train Fourier-CPPN (compare Gaussian scales)

In [ ]:
def train_models_for_scales(
    *,
    scales: List[float],
    epochs: int,
    lr: float,
    device: str | torch.device,
    loader: DataLoader,
    latent_dim: int,
    c_max: int,
    mapping_size: int,
    in_dim: int,
 ):
    models: Dict[float, torch.nn.Module] = {}
    for s in scales:
        print(f"\n=== Training gauss_scale={s} ===")
        model = train_utils.train_fourier_cppn_for_trajectory(
            trajectory=trajectories[0],  # kept for API compatibility; training uses `loader`
            latent_dim=latent_dim,
            c_max=c_max,
            gauss_scale=float(s),
            mapping_size=mapping_size,
            device=str(device),
            epochs=int(epochs),
            lr=float(lr),
            loader=loader,
            print_every=max(1, int(epochs) // 5),
            in_dim=in_dim,
        )
        models[float(s)] = model
    return models

scales = [0.01, 0.1, 1.0, 2.0, 5.0]
epochs = int(hparams.epochs)
lr = float(hparams.lr)

models_by_scale = train_models_for_scales(
    scales=scales,
    epochs=epochs,
    lr=lr,
    device=device,
    loader=dl,
    latent_dim=int(trajectories[0].shape[1]),
    c_max=int(hparams.c_max),
    mapping_size=int(hparams.mapping_size),
    in_dim=2,
 )

## 6) Dense Grid Inference (2D field of latents)

In [ ]:
# Build a normalized 2D pixel grid in [0,1]^2 -> (H*W, 2)
H, W = 1024, 1024  # resolution (height, width)
xs = torch.linspace(0.0, 1.0, W, device=device, dtype=torch.float32)
ys = torch.linspace(0.0, 1.0, H, device=device, dtype=torch.float32)
yy, xx = torch.meshgrid(ys, xs, indexing="ij")
coords_xy = torch.stack((xx, yy), dim=-1).reshape(-1, 2)
coords_xy.shape

In [ ]:
coords_dataset = torch.utils.data.TensorDataset(coords_xy)
coords_dl = DataLoader(coords_dataset, batch_size=256, shuffle=False, drop_last=False)
len(coords_dataset), coords_dl.batch_size

In [ ]:
def predict_grid(
    model: torch.nn.Module,
    *,
    coords_dl: DataLoader,
    H: int,
    W: int,
    model_device: str | torch.device,
 ):
    model.eval()
    outs: List[torch.Tensor] = []
    with torch.no_grad():
        for (batch_coords,) in coords_dl:
            outs.append(model(batch_coords.to(model_device)))
    return torch.cat(outs, dim=0).view(H, W, -1)

# Run every trained model over the full grid
model_device = device
preds_by_scale: Dict[float, torch.Tensor] = {}
for s, m in models_by_scale.items():
    preds_by_scale[s] = predict_grid(m, coords_dl=coords_dl, H=H, W=W, model_device=model_device)
    print(f"scale={s}: preds grid {tuple(preds_by_scale[s].shape)}")

In [ ]:
# (moved into `preds_by_scale` above)
sorted(preds_by_scale.keys())

In [ ]:
# Convenience: stack predictions in a stable sigma order
sigma_order = sorted(preds_by_scale.keys())
preds = [preds_by_scale[s] for s in sigma_order]
sigma_order

In [ ]:
# `preds` already prepared above as [preds_by_scale[s] for s in sigma_order]
len(preds), sigma_order

## 7) Visualize (control curves + latent channel across σ)

In [ ]:
def plot_sigma_latent_channel(
    *,
    curves: List[torch.Tensor],
    preds_by_sigma: Dict[float, torch.Tensor],
    channel_idx: int = 0,
    vmin: float = -3.0,
    vmax: float = 3.0,
 ):
    sigmas = sorted(preds_by_sigma.keys())
    if len(sigmas) == 0:
        raise ValueError("preds_by_sigma is empty")

    # 1 axis for control space + 1 axis per sigma + 1 for colorbar
    n_img = 1 + len(sigmas)
    fig, axes = plt.subplots(
        1, n_img + 1,
        figsize=(3.3 * (n_img + 1), 3.8),
        dpi=200,
        gridspec_kw={"width_ratios": [1] * n_img + [0.05], "wspace": 0.1},
    )
    ax_ctrl = axes[0]
    ax_imgs = axes[1:n_img]
    cax = axes[n_img]

    # Control-space trajectories
    cmap = plt.get_cmap("tab20")
    colors = cmap(np.linspace(0, 1, max(1, len(curves))))
    for j, pts in enumerate(curves):
        p = pts.detach().cpu().numpy()
        ax_ctrl.plot(p[:, 0], p[:, 1], "-", lw=1.2, color=colors[j % len(colors)])
    ax_ctrl.set_xlim(-0.02, 1.02)
    ax_ctrl.set_ylim(-0.02, 1.02)
    ax_ctrl.set_aspect("equal", adjustable="box")
    ax_ctrl.set_title("Control Space")

    # Latent channel images
    im = None
    for ax, s in zip(ax_imgs, sigmas):
        grid = preds_by_sigma[s]  # (H, W, D)
        if channel_idx < 0 or channel_idx >= grid.shape[-1]:
            raise IndexError(f"channel_idx={channel_idx} out of range for D={grid.shape[-1]}")
        ch = grid[..., channel_idx].detach().cpu().numpy()
        im = ax.imshow(ch, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
        ax.set_title(f"σ = {s}")
        ax.axis("off")

    if im is not None:
        fig.colorbar(im, cax=cax)
        plt.tight_layout()
        # shrink colorbar a bit + vertically center
        pos = cax.get_position()
        new_h = pos.height * 0.74
        new_y0 = pos.y0 + (pos.height - new_h) / 2.0
        cax.set_position([pos.x0, new_y0, pos.width, new_h])

    plt.show()

plot_sigma_latent_channel(curves=curves, preds_by_sigma=preds_by_scale, channel_idx=0)